# Stage 1 — Dataset EDA: Montgomery/Shenzhen CXR with lung masks

**Learning goals**
- Load DICOM-era chest X-rays + binary masks with Python
- Understand image shape/range conventions (lung CXRs are often inverted: dark lungs = air)
- Quantify mask statistics (how much of the image is lung?)
- Overlay masks on images — the single most important sanity check in segmentation

**Refresher before you run:** in segmentation we predict a class *per pixel*. The metrics are Dice and IoU:
- Dice = 2|A∩B| / (|A|+|B|)  — fraction of twice-overlapped pixels
- IoU = |A∩B| / |A∪B|

TODO: run all cells, then answer the questions marked **❓**.

In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

RNG = np.random.default_rng(42)
DATA = Path("../data")
print(DATA.resolve())

In [ ]:
# Discover what we actually got — directory tree, first two levels
subdirs = sorted(p for p in DATA.rglob("*") if p.is_dir())
for d in subdirs[:25]:
    n = len(list(d.iterdir()))
    print(f"{d.relative_to(DATA)}/  ({n} entries)")

### Dataset layout (verified — from your `data/` folder)

```
data/
├── Montgomery/MontgomerySet/
│   ├── CXR_png/          138 CXRs — MCUCXR_*.png, big: ~4020×4892, uint8
│   ├── ManualMask/
│   │   ├── leftMask/     binary masks (load as True/False!)
│   │   └── rightMask/    counts may differ by ±1 → your EDA finds it
│   └── ClinicalReadings/ age/gender/TB status per image (*.txt)
├── ChinaSet_AllFiles/ChinaSet_AllFiles/
│   ├── CXR_png/          662 CXRs — CHNCXR_*.png, ~2919×3000, uint8
│   └── ClinicalReadings/
└── (ignore: __MACOSX junk)
```

**NLM conventions to know:**
- CXRs are high-res and **stored inverted** vs typical radiography (bright ↔ dark); check pixel means per set.
- Filename suffix `_0` / `_1` = TB-negative / TB-positive (derivable from ClinicalReadings).
- Only Montgomery ships masks in this Kaggle upload → supervised segmentation trains on ~138 pairs.

### Pair up images and masks

**❓ Q1:** filenames of the mask vs the image differ how? Rule: image `CXR_png/X.png` ↔ masks `ManualMask/leftMask/X.png` + `rightMask/X.png`, and the **whole-lung mask = left ∪ right** (pixel-wise max). Verify, then implement.

In [ ]:
# TODO: adapt the suffix/suffixes to the real layout you observe above
IMAGE_SUFFIXES = (".png", ".jpg", ".dcm")

def is_mask(path: Path) -> bool:
    """Return True if a file looks like a ground-truth lung mask."""
    raise NotImplementedError("rule from Q1")

# every image file in the dataset, with optional set label (Montgomery / Shenzhen)
all_files = sorted(p for p in DATA.rglob("*") if p.suffix.lower() in IMAGE_SUFFIXES)
# TODO: build (image_path, mask_path) pairs; use Counter to verify each set matches N images <-> N masks

### Image properties table
Collect per-image: source set, width, height, pixel min/max/mean, and lung pixel fraction from the mask. Put it in a `pandas` DataFrame — this is EDA you already know how to do.

In [ ]:
def load_grayscale(path: Path) -> np.ndarray:
    """Load image as float32 [0, 1] grayscale. Handle PNG (uint8) and DICOM if present."""
    raise NotImplementedError

In [ ]:
# TODO: loop over pairs, filling the rows table below
# rows = [{"set": ..., "file": ..., "w": ..., "h": ..., "min": ..., "max": ..., "lung_frac": ...}]
# df = pd.DataFrame(rows); df.describe per set with df.groupby("set")[...]

**❓ Q2:** are image sizes uniform or mixed? What resolution will we need to resize to for a fixed-size U-Net (hint: most U-Nets want dims divisible by 2^4 = 16)?

**❓ Q3:** what fraction of a CXR is actually lung? Typical answer is ~30–40% — if you see far less, suspect empty masks or inverted values.

### The sanity check: overlay a few (image, mask) pairs

In [ ]:
# TODO: pick 4 random pairs per set and plot
# fig, axes = plt.subplots(2, 4, figsize=(16, 8))
# each panel: show image in gray, mask as red with alpha≈0.4
# look for misalignment, inverted masks, or clipped lungs

**❓ Q4:** look at the masks alone (schema-check): unique values only in {0, 1}? (
Some mask files use 0/255.) If 0/255, where should we normalize — in a helper in `src/data.py` so every later notebook inherits the fix?

**❓ Q5 (reflection):** given these masks, what would a *Dice score of 0.75* look like visually? Answer before Stage 2 — then check yourself against your classical baseline results.